# MarketSense Model B

Compare the 18-feature Model A baseline with the same baseline plus headline sentiment or headline-and-lead sentiment. All models use the same purged validation and test rows. Copy `model_b_dataset.parquet` and `model_b_meta.json` to the Drive folder below, choose upload, or run locally from the repository root.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {"sklearn": "scikit-learn", "xgboost": "xgboost",
            "joblib": "joblib", "pyarrow": "pyarrow"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    importlib.invalidate_caches()

import json, re, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import joblib, sklearn, xgboost as xgb
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 200, "display.max_columns", 30)
print("pandas", pd.__version__, "| sklearn", sklearn.__version__, "| xgboost", xgb.__version__)

## Configuration and data

In [ ]:
CFG = dict(
    DATA_SOURCE="drive",                                    # "drive" | "upload" | "local"
    DRIVE_DIR="/content/drive/MyDrive/MarketSense/model_b",  # folder holding the two exported files
    LOCAL_DIR="data/model_b",
    OUT_DIR="/content/model_b_artifacts",
    TRAIN_FRAC=0.70,                 # by unique DATES, chronological; the remainder after VAL is test
    VAL_FRAC=0.15,
    DROP_CORP_ACTION_SUSPECT=True,   # drop rows whose label window contains a raw |1-day move| above the export threshold
    CLIP_PCT=(0.5, 99.5),            # winsorise bounds (learned on TRAIN only) used for logistic regression
    BOOTSTRAP_REPS=300, BOOTSTRAP_BLOCK_DATES=10,
    SEED=42,
)


In [ ]:
def get_data_dir() -> Path:
    if CFG["DATA_SOURCE"] == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        return Path(CFG["DRIVE_DIR"])
    if CFG["DATA_SOURCE"] == "upload":
        from google.colab import files
        up = files.upload()                       # select model_b_dataset.parquet AND model_b_meta.json
        d = Path("/content/model_b_upload"); d.mkdir(exist_ok=True)
        for name, data in up.items():
            (d / re.sub("[ ]?[(][0-9]+[)]", "", name)).write_bytes(data)   # strip browser (1) suffixes
        return d
    return Path(CFG["LOCAL_DIR"])

DATA_DIR = get_data_dir()
ds = pd.read_parquet(DATA_DIR / "model_b_dataset.parquet")
meta = json.loads((DATA_DIR / "model_b_meta.json").read_text())

FEATURES, TARGET, RET, HORIZON = meta["features"], meta["target_col"], meta["return_col"], meta["horizon_sessions"]
ds["trading_date"] = pd.to_datetime(ds["trading_date"]); ds["label_end_date"] = pd.to_datetime(ds["label_end_date"])

sentiment = meta["sentiment_features"]
FEATURE_SETS = {
    "model_a": FEATURES,
    "model_b_headline": FEATURES + sentiment["hl"],
    "model_b_headline_lead": FEATURES + sentiment["hl_lead"],
}
NAMES = {
    "model_a": "Model A",
    "model_b_headline": "Model B (headline-only)",
    "model_b_headline_lead": "Model B (headline + lead)",
}
assert len(ds) == 132_899 and ds.symbol.nunique() == 94 and HORIZON == 5
assert len(FEATURES) == 18 and all(len(group) == 36 for group in sentiment.values())
all_features = FEATURES + sentiment["hl"] + sentiment["hl_lead"]
assert len(set(all_features)) == len(all_features)
assert np.isfinite(ds[all_features].to_numpy(float)).all(), "NaN/inf in features"
assert set(FEATURES) <= set(ds.columns), "feature columns missing from dataset"
assert not any(k in f for f in FEATURES for k in ("target", "forward", "label", "symbol", "stock_id", "date")), "outcome/identifier in features"
assert ds[FEATURES].notna().all().all() and np.isfinite(ds[FEATURES].to_numpy(float)).all(), "NaN/inf in features"
assert set(ds[TARGET].unique()) <= {0, 1}, "target must be 0/1"
assert ((ds[RET] > 0).astype(int) == ds[TARGET]).all(), "target inconsistent with forward return"
assert not ds.duplicated(["symbol", "trading_date"]).any(), "duplicate (symbol, date) rows"
assert (ds["label_end_date"] > ds["trading_date"]).all(), "label must end after prediction date"
print(f"rows={len(ds):,}  stocks={ds.symbol.nunique()}  dates={ds.trading_date.min().date()} -> {ds.trading_date.max().date()}")
print("features:", len(FEATURES), "| corp-action-suspect rows:", int(ds.corp_action_suspect_5d.sum()))
print("overall share up:", round(ds[TARGET].mean(), 4), "| exact-zero returns (labelled 0):", int((ds[RET] == 0).sum()))
if CFG["DROP_CORP_ACTION_SUSPECT"]:
    ds = ds[~ds.corp_action_suspect_5d].reset_index(drop=True)
    print("after dropping suspect rows:", f"{len(ds):,}")

## Chronological split and 5-session label purging

In [ ]:
dates = np.sort(ds["trading_date"].unique())
val_start  = pd.Timestamp(dates[int(len(dates) * CFG["TRAIN_FRAC"])])
test_start = pd.Timestamp(dates[int(len(dates) * (CFG["TRAIN_FRAC"] + CFG["VAL_FRAC"]))])

train = ds[(ds.trading_date < val_start) & (ds.label_end_date < val_start)]
val   = ds[(ds.trading_date >= val_start) & (ds.trading_date < test_start) & (ds.label_end_date < test_start)]
test  = ds[ds.trading_date >= test_start]

assert train.label_end_date.max() < val.trading_date.min(), "train labels overlap validation"
assert val.label_end_date.max() < test.trading_date.min(), "validation labels overlap test"

rows = []
for name, d in [("train", train), ("val", val), ("test", test)]:
    rows.append(dict(split=name, rows=len(d), stocks=d.symbol.nunique(), start=d.trading_date.min().date(),
                     end=d.trading_date.max().date(), share_up=round(d[TARGET].mean(), 4)))
print(pd.DataFrame(rows).to_string(index=False))
print("rows purged at the boundaries:", len(ds[ds.trading_date < test_start]) - len(train) - len(val))

## Fit and compare

In [ ]:
def metrics(y, proba, thr=0.5):
    pred = (np.asarray(proba) >= thr).astype(int)
    auc = roc_auc_score(y, proba) if len(np.unique(proba)) > 1 and len(np.unique(y)) > 1 else 0.5
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(thr=round(float(thr), 3), accuracy=accuracy_score(y, pred), balanced_acc=balanced_accuracy_score(y, pred),
                precision=precision_score(y, pred, zero_division=0), recall=recall_score(y, pred, zero_division=0),
                f1=f1_score(y, pred, zero_division=0), roc_auc=auc, tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

def tune_threshold(y, proba):
    """Pick the threshold on VALIDATION that maximises balanced accuracy (ties -> closest to 0.5)."""
    grid = np.round(np.arange(0.35, 0.651, 0.005), 3)
    best = max(grid, key=lambda t: (round(balanced_accuracy_score(y, (proba >= t).astype(int)), 6), -abs(t - 0.5)))
    return float(best)

In [ ]:
SPLITS = {"val": val, "test": test}
y_tr = train[TARGET].to_numpy()
y_val = val[TARGET].to_numpy()
weights = float((y_tr == 0).sum() / (y_tr == 1).sum())
clip_lo, clip_hi = np.percentile(train[FEATURES], CFG["CLIP_PCT"], axis=0)


def logistic_input(frame, features):
    # Retain sparse nonzero sentiment values. Apply Model A's clipping to its price features.
    data = frame[features].copy()
    data[FEATURES] = np.clip(data[FEATURES].to_numpy(), clip_lo, clip_hi)
    return data


models, thresholds, predictions, results = {}, {}, {}, {}
for name, features in FEATURE_SETS.items():
    logistic = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(C=0.1, max_iter=2000, class_weight="balanced", random_state=CFG["SEED"]),
    )
    logistic.fit(logistic_input(train, features), y_tr)

    boosted = xgb.XGBClassifier(
        n_estimators=2000, learning_rate=0.02, max_depth=3, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=10, reg_alpha=0,
        tree_method="hist", objective="binary:logistic", eval_metric="auc",
        early_stopping_rounds=100, scale_pos_weight=weights, random_state=CFG["SEED"], n_jobs=-1,
    )
    boosted.fit(train[features], y_tr, eval_set=[(val[features], y_val)], verbose=False)
    for family, estimator in {"logistic_regression": logistic, "xgboost": boosted}.items():
        key = (family, name)
        models[key] = estimator
        by_split = {}
        for split, frame in SPLITS.items():
            inputs = logistic_input(frame, features) if family == "logistic_regression" else frame[features]
            by_split[split] = estimator.predict_proba(inputs)[:, 1]
        threshold = tune_threshold(y_val, by_split["val"])
        thresholds[key] = threshold
        predictions[key] = by_split
        results[key] = {
            split: metrics(frame[TARGET].to_numpy(), by_split[split], threshold)
            for split, frame in SPLITS.items()
        }

primary_family = max(("logistic_regression", "xgboost"),
                     key=lambda family: results[(family, "model_a")]["val"]["roc_auc"])
print(f"Primary comparison uses {primary_family}, selected by Model A validation AUC.")
comparison = pd.DataFrame([
    {"estimator": family, "model": NAMES[name], "split": split,
     **{k: report[k] for k in ("roc_auc", "balanced_acc", "accuracy", "f1", "thr")}}
    for (family, name), by_split in results.items()
    for split, report in by_split.items()
])
print(comparison.round(4).to_string(index=False))

## Moving-block confidence intervals and paired AUC tests

Each replicate samples contiguous blocks of trading dates and takes all available stocks on those dates. Every model uses the same sampled rows. The two-sided p-value compares the observed AUC difference with the centered bootstrap distribution under a zero-difference null.

In [ ]:
def bootstrap_indices(frame, reps, width, seed):
    by_date = frame.groupby("trading_date", sort=True).indices
    dates = sorted(by_date)
    width = min(width, len(dates))
    blocks = int(np.ceil(len(dates) / width))
    rng = np.random.default_rng(seed)
    for _ in range(reps):
        starts = rng.integers(0, len(dates) - width + 1, blocks)
        positions = (starts[:, None] + np.arange(width)).ravel()[:len(dates)]
        yield np.concatenate([by_date[dates[i]] for i in positions])


def ci95(values):
    return [float(v) for v in np.percentile(values, [2.5, 97.5])]


intervals, paired = [], []
for split, frame in SPLITS.items():
    y = frame[TARGET].to_numpy()
    aucs = {key: [] for key in models}
    balances = {key: [] for key in models}
    for idx in bootstrap_indices(frame, CFG["BOOTSTRAP_REPS"], CFG["BOOTSTRAP_BLOCK_DATES"], CFG["SEED"]):
        if np.unique(y[idx]).size < 2:
            continue
        for key in models:
            proba = predictions[key][split][idx]
            aucs[key].append(roc_auc_score(y[idx], proba))
            balances[key].append(balanced_accuracy_score(y[idx], proba >= thresholds[key]))
    for (family, name), values in aucs.items():
        key = (family, name)
        intervals.append(dict(estimator=family, model=NAMES[name], split=split,
                              auc_ci95=ci95(values), balanced_acc_ci95=ci95(balances[key])))
        if name == "model_a":
            continue
        baseline = (family, "model_a")
        observed = results[key][split]["roc_auc"] - results[baseline][split]["roc_auc"]
        differences = np.asarray(values) - np.asarray(aucs[baseline])
        extremes = np.count_nonzero(np.abs(differences - observed) >= abs(observed))
        paired.append(dict(estimator=family, model=NAMES[name], split=split, delta_auc=observed,
                           delta_auc_ci95=ci95(differences),
                           p_value=float((extremes + 1) / (len(differences) + 1)),
                           bootstrap_replicates=len(differences)))

print(pd.DataFrame(intervals).to_string(index=False))
print(pd.DataFrame(paired).to_string(index=False))

## Save estimators, predictions, and evaluation results

In [ ]:
OUT = Path(CFG["OUT_DIR"]) if CFG["DATA_SOURCE"] != "local" else DATA_DIR / "artifacts"
OUT.mkdir(parents=True, exist_ok=True)
artifacts = {}
for (family, name), estimator in models.items():
    key = (family, name)
    stem = f"{name}_{family}"
    features = FEATURE_SETS[name]
    if family == "logistic_regression":
        path = OUT / f"{stem}.joblib"
        joblib.dump(estimator, path)
        restored = joblib.load(path).predict_proba(logistic_input(test, features))[:, 1]
    else:
        path = OUT / f"{stem}.json"
        estimator.save_model(path)
        restored_model = xgb.XGBClassifier()
        restored_model.load_model(path)
        restored = restored_model.predict_proba(test[features])[:, 1]
    np.testing.assert_allclose(restored, predictions[key]["test"])
    artifacts[stem] = dict(file=path.name, features=features, threshold=thresholds[key],
                           metrics=results[key])

for split, frame in SPLITS.items():
    exported = frame[["symbol", "trading_date", "label_end_date", TARGET, RET]].copy()
    for (family, name), by_split in predictions.items():
        exported[f"{name}_{family}_prob_up"] = by_split[split]
    exported.to_parquet(OUT / f"{split}_predictions.parquet", index=False)

summary = dict(config=CFG, primary_family=primary_family, artifacts=artifacts,
               clipping=dict(features=FEATURES, lower=clip_lo.tolist(), upper=clip_hi.tolist()),
               split_dates=dict(validation=str(val_start.date()), test=str(test_start.date())),
               split_rows=dict(train=len(train), val=len(val), test=len(test)),
               versions=dict(sklearn=sklearn.__version__, xgboost=xgb.__version__),
               dataset_meta=meta, intervals=intervals, paired_auc=paired)
(OUT / "evaluation.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
comparison.to_csv(OUT / "comparison.csv", index=False)

if CFG["DATA_SOURCE"] == "drive":
    destination = DATA_DIR / "artifacts"
    shutil.copytree(OUT, destination, dirs_exist_ok=True)
    print(f"Saved artifacts to {destination}")
elif CFG["DATA_SOURCE"] == "upload":
    from google.colab import files
    files.download(shutil.make_archive(str(OUT), "zip", OUT))
else:
    print(f"Saved artifacts to {OUT}")